# 07 -- Web UI Walkthrough

RankFlow ships with an interactive Streamlit web UI for browsing and comparing experiments. This notebook:

1. Generates realistic synthetic experiment data (three pipeline variants)
2. Saves them to an experiment store
3. Explains every UI page and what you can do with it

By the end you will have a ready-to-use experiment store and know exactly what to expect when you launch `rankflow ui`.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np

from rankflow import (
    RankFlow,
    Experiment,
    ExperimentStore,
    compare_experiments,
)

---
## Step 1: Generate synthetic experiment data

We'll simulate a realistic scenario: three pipeline configurations evaluated on 30 queries, each returning 20 candidate documents.

| Experiment | Retriever | Reranker | Chunk size | Expected quality |
|---|---|---|---|---|
| `bm25-baseline` | BM25 | none | 512 | Low |
| `cross-encoder-v1` | BM25 | cross-encoder (MiniLM) | 512 | Medium |
| `cross-encoder-v2` | BM25 | cross-encoder (large) | 256 | Highest |

In [ ]:
N_QUERIES = 30
N_DOCS = 20
STEP_LABELS = ["BM25", "Reranker"]
CHUNK_LABELS = [f"doc_{i}" for i in range(N_DOCS)]


def generate_rankflows(
    seed: int,
    reranker_strength: float = 0.0,
    n_relevant: int = 3,
) -> list[RankFlow]:
    """Generate RankFlow objects simulating a retrieval pipeline.

    Args:
        seed: Random seed for reproducibility.
        reranker_strength: How much the reranker improves results (0 = no
            improvement, higher = better). Controls noise added to shuffle.
        n_relevant: Number of relevant documents per query.
    """
    rng = np.random.default_rng(seed)
    rankflows = []

    for q in range(N_QUERIES):
        # Step 0: BM25 retrieval (random initial ranking)
        bm25_ranks = rng.permutation(N_DOCS)

        # Step 1: Reranker -- shuffle with noise, less noise = better reranker
        noise_scale = max(0.5, 5.0 - reranker_strength)
        reranked = np.argsort(np.argsort(
            bm25_ranks.astype(float) + rng.normal(0, noise_scale, N_DOCS)
        ))

        ranks = np.array([bm25_ranks, reranked])

        # Pick relevant documents (biased toward lower indices to be realistic)
        weights = np.exp(-np.arange(N_DOCS) / 5.0)
        weights /= weights.sum()
        relevant = rng.choice(
            CHUNK_LABELS, size=n_relevant, replace=False, p=weights
        ).tolist()

        # Generate relevance grades (3 = highly relevant, 1 = marginally)
        grades = {doc: int(rng.integers(1, 4)) for doc in relevant}

        # Generate fake scores
        bm25_scores = rng.uniform(0.2, 1.0, N_DOCS)
        reranker_scores = rng.uniform(0.3, 1.0, N_DOCS)
        scores = np.array([bm25_scores, reranker_scores])

        rf = RankFlow(
            ranks=ranks,
            step_labels=STEP_LABELS,
            chunk_labels=CHUNK_LABELS,
            relevant_chunks=relevant,
            relevance_grades=grades,
            scores=scores,
        )
        rf.query_label = f"query_{q:02d}"
        rankflows.append(rf)

    return rankflows


baseline_rfs = generate_rankflows(seed=42, reranker_strength=0.0)
v1_rfs = generate_rankflows(seed=42, reranker_strength=2.0)
v2_rfs = generate_rankflows(seed=42, reranker_strength=4.0)

print(f"Generated {N_QUERIES} queries x 3 experiments")

---
## Step 2: Save experiments to a store

An `ExperimentStore` is just a directory. Each experiment is one JSON file.

In [ ]:
# Use a temp directory for the tutorial -- replace with your own path in practice
STORE_PATH = Path(tempfile.mkdtemp()) / "my_experiments"
store = ExperimentStore(STORE_PATH)

store.save(Experiment(
    name="bm25-baseline",
    config={
        "retriever": "bm25",
        "top_k": 100,
        "reranker": "none",
        "chunk_size": 512,
    },
    rankflows=baseline_rfs,
    tags=["baseline", "v1"],
    description="BM25 retrieval with no reranking",
))

store.save(Experiment(
    name="cross-encoder-v1",
    config={
        "retriever": "bm25",
        "top_k": 100,
        "reranker": "cross-encoder",
        "model": "ms-marco-MiniLM-L-6-v2",
        "chunk_size": 512,
    },
    rankflows=v1_rfs,
    tags=["challenger", "v1"],
    description="BM25 + MiniLM cross-encoder reranker",
))

store.save(Experiment(
    name="cross-encoder-v2",
    config={
        "retriever": "bm25",
        "top_k": 100,
        "reranker": "cross-encoder",
        "model": "ms-marco-MiniLM-L-12-v2",
        "chunk_size": 256,
    },
    rankflows=v2_rfs,
    tags=["challenger", "v2"],
    description="BM25 + large cross-encoder, smaller chunks",
))

print(f"Store path: {STORE_PATH}")
print(f"Files: {[f.name for f in STORE_PATH.iterdir()]}")

---
## Step 3: Verify the store works

Before launching the UI, let's make sure the data round-trips correctly.

In [ ]:
print("All experiments in store:")
for info in store.list():
    tags = ", ".join(info["tags"])
    print(f"  {info['name']:25s} | {info['n_queries']:2d} queries | tags: {tags}")

# Filter by tag
challengers = store.list(tag="challenger")
print(f"\nChallenger experiments: {[e['name'] for e in challengers]}")

In [ ]:
# Load one experiment and check headline metrics
loaded = store.load("cross-encoder-v2")
print(f"Loaded: {loaded.name} ({loaded.n_queries} queries)")
print(f"Config: {loaded.config}")

summary = loaded.metrics_summary(k=10)
print("\nHeadline metrics (mean across queries):")
for metric, value in summary.items():
    print(f"  {metric}: {value:.3f}")

---
## Step 4: Quick comparison before launching the UI

The UI wraps `compare_experiments()` -- let's see what it produces so you know what to expect.

In [ ]:
baseline = store.load("bm25-baseline")
challenger = store.load("cross-encoder-v2")

report = compare_experiments(baseline, challenger, k=10)

print(f"Baseline:   {report.baseline_name}")
print(f"Challenger: {report.challenger_name}")
print(f"\nWin / Loss / Tie: {report.wins}W / {report.losses}L / {report.ties}T")
print(f"Win rate: {report.win_rate:.0%}")

print(f"\n{'Metric':<20s} {'Delta':>8s} {'p-value':>8s} {'Sig?':>5s}")
print("-" * 45)
for metric, data in report.metric_deltas.items():
    sig = "*" if data['p_value'] < 0.05 else ""
    print(f"{metric:<20s} {data['delta']:>+8.3f} {data['p_value']:>8.3f} {sig:>5s}")

---
## Step 5: Launch the Web UI

Now that the experiment store is populated, launch the UI by running this in your terminal:

```bash
pip install rankflow[ui]   # installs streamlit
rankflow ui /path/to/my_experiments
```

Replace `/path/to/my_experiments` with the store path printed above.

Alternatively, run Streamlit directly:

```bash
streamlit run $(python -c "import rankflow.ui.app; print(rankflow.ui.app.__file__)") -- /path/to/my_experiments
```

The UI opens in your browser at `http://localhost:8501`.

In [ ]:
# Print the command you would run (copy-paste it into your terminal)
print(f"rankflow ui {STORE_PATH}")

---
## UI Page Guide

The sidebar has four pages. Here is what each one does and when to use it.

### Page 1: Experiments

**What it shows:**
- A table of all saved experiments with name, number of queries, tags, config summary, and timestamp.
- A tag filter in the sidebar to narrow down experiments (e.g., show only `baseline` tag).
- A "Preview experiment" dropdown with a "Load metrics" button that computes headline metrics (NDCG, MRR, P@K, R@K, MAP) as cards.

**When to use it:**
- First thing after launching the UI -- get an overview of all experiments in the store.
- Quickly check if a new experiment has been saved correctly.
- Filter by tag to find all experiments from a specific iteration or team.

### Page 2: Compare

**What it shows:**
- Two dropdowns to select baseline and challenger experiments.
- A sidebar slider to set K (top-K for metric computation).
- After clicking "Compare":
  - **Configuration Differences** -- table showing only parameters that changed between the two configs.
  - **Metric Comparison** -- cards showing each metric's challenger value with delta and significance marker (checkmark if p < 0.05).
  - **Win / Loss / Tie** -- count + progress bar showing win rate.
  - **Per-Query Details** -- full table with baseline/challenger/delta for every metric on every query.

**When to use it:**
- The main decision-making view: "did config B beat config A?"
- Check statistical significance before deciding to ship a config change.
- Look at the config diff to understand exactly what changed.

### Page 3: Query Explorer

**What it shows:**
- A dropdown to select an experiment.
- A table listing every query with its step count, document count, and final-step metrics (NDCG@K, MRR, P@K).
- A query selector for drill-down: selecting a query renders its **rank evolution plot** (the core RankFlow visualization) inline.

**When to use it:**
- After comparison reveals regressions: drill into the specific query that got worse.
- Spot-check individual queries to understand *why* metrics moved.
- Visual debugging: see which documents were promoted/demoted by the reranker.

### Page 4: Deep Dive

**What it shows:**
- A dropdown to select an experiment.
- **Configuration** -- the full config dict as JSON.
- **Metrics Dashboard** -- box plots for all metrics across all queries (via `BatchRankFlow.plot_dashboard()`).
- **Metric Evolution** -- line plot with error bars showing how metrics change from step to step.
- **Win / Loss Analysis** -- table showing win/loss/tie counts at each step transition.
- **Failure Cases** -- table of queries where the final step made metrics worse (threshold = -0.05).

**When to use it:**
- Deep analysis of a single experiment's behavior.
- Understand metric variance across queries (box plots).
- Find failure cases that need investigation.

---
## Typical workflow

A realistic retrieval tuning session follows this pattern:

```
1. Run pipeline with config A  -->  save as Experiment("config-a", ...)
2. Run pipeline with config B  -->  save as Experiment("config-b", ...)
3. Launch UI:  rankflow ui ./experiments
4. Experiments page: check both are there, preview metrics
5. Compare page: select A vs B, check significance
6. Query Explorer: drill into regressions
7. Deep Dive: check variance and failure cases for config B
8. Iterate: adjust config, run again, save as Experiment("config-c", ...)
```

Each iteration adds ~30 seconds of overhead (save + reload UI). No database, no cloud service, no YAML configs -- just Python dicts and JSON files.

---
## Bonus: Programmatic comparison (no UI needed)

Everything the UI does is available as a Python API. You can script comparisons in CI or Jupyter.

In [ ]:
# Compare all challengers against the baseline
baseline = store.load("bm25-baseline")

for info in store.list(tag="challenger"):
    challenger = store.load(info["name"])
    report = compare_experiments(baseline, challenger, k=10)

    ndcg = report.metric_deltas.get("ndcg_at_k", {})
    delta = ndcg.get("delta", 0)
    p = ndcg.get("p_value", 1)
    sig = "SIGNIFICANT" if p < 0.05 else "not sig."

    print(
        f"{info['name']:25s} | "
        f"NDCG delta={delta:+.3f} (p={p:.3f}, {sig}) | "
        f"{report.wins}W/{report.losses}L/{report.ties}T"
    )

In [ ]:
# Find the worst regressions across the best challenger
best = store.load("cross-encoder-v2")
report = compare_experiments(baseline, best, k=10)

regressions = report.regression_queries("ndcg_at_k")
print(f"\nRegressions in cross-encoder-v2 vs baseline: {len(regressions)} queries")

if regressions:
    sorted_reg = sorted(regressions, key=lambda q: q["ndcg_at_k_delta"])
    print("\nWorst 3:")
    for q in sorted_reg[:3]:
        print(
            f"  {q['query_label']}: "
            f"NDCG {q['ndcg_at_k_baseline']:.3f} -> {q['ndcg_at_k_challenger']:.3f} "
            f"(delta={q['ndcg_at_k_delta']:+.3f})"
        )

In [ ]:
# Drill into the worst regression
if regressions:
    worst_label = sorted_reg[0]["query_label"]
    worst_idx = [rf.query_label for rf in best.rankflows].index(worst_label)

    print(f"Rank evolution for {worst_label} (cross-encoder-v2):")
    best.rankflows[worst_idx].plot()

---
## Cleanup

The temp directory will be cleaned up automatically. In practice, use a persistent path:

```python
store = ExperimentStore("./experiments")  # persists across sessions
```

In [ ]:
# Confirm the store is still valid
assert store.exists("bm25-baseline")
assert store.exists("cross-encoder-v1")
assert store.exists("cross-encoder-v2")
print(f"Store at {STORE_PATH} contains {len(store.list())} experiments. Ready for UI.")

---

**Previous:** [06 -- Experiments and Comparison](06_experiments_and_comparison.ipynb)

**Start from the beginning:** [01 -- Quick Start](01_quickstart.ipynb)